# CareFlow on Colab A100

Runs any CareFlow experiment on vLLM servers with concurrent requests (10–50× the throughput of the M5 Pro).
**Read `docs/colab.md` first.** Key rules:
- Colab runs use the original HF checkpoints (not MLX 4-bit): **re-validate the judge here first** (cell 6) and give every run its own `experiment.name` (`*_colab`). Never mix Colab and Mac numbers in one table; latency from Colab is not reported.
- The LLM cache lives on local disk (`/content/CareFlow/.cache/llm`) and is **snapshotted to Drive after every run** (SQLite must not be opened on Drive directly). A dropped session loses at most the current run, and a rerun replays everything already cached.
- Generator and judge do not fit on one 40 GB A100 together → every `systems_eval` run is **two phases**: generation (gen servers) → judging (judge server; generations replay from the cache).

Secrets (Colab → 🔑): `GH_TOKEN` (read access to the repo), `HF_TOKEN`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`, `S3_ACSESS_KEY_ID`, `S3_SECRET_ACSESS_KEY` (DagsHub DVC).

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess

mem_gb = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).split()[0]) / 1024
JUDGE_ARGS = "" if mem_gb > 70 else "--quantization fp8"   # Gemma-4-26B-A4B bf16 ≈ 52 GB
GEN_ARGS = "" if mem_gb > 70 else "--quantization fp8"     # Qwen3.5-9B next to the 2B on 40 GB
print(f"{mem_gb:.0f} GB -> JUDGE_ARGS={JUDGE_ARGS!r} GEN_ARGS={GEN_ARGS!r}")

## 2. Drive, secrets, repo

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/careflow"; os.makedirs(DRIVE, exist_ok=True)
for k in ["HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD", "S3_ACSESS_KEY_ID", "S3_SECRET_ACSESS_KEY"]:
    try:
        os.environ[k] = userdata.get(k)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print("missing secret", k)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
BRANCH = "main"   # the branch with the code you want to run (commit + push from the Mac first)
!git clone -q -b {BRANCH} https://{userdata.get('GH_TOKEN')}@github.com/noviyblock/CareFlow.git /content/CareFlow || (cd /content/CareFlow && git pull -q)
%cd /content/CareFlow
!git log --oneline -1

## 3. Environments: project (uv, no MLX) + vLLM (separate venv, its own torch)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv sync -q --extra api --extra dev
!python -m venv /content/vllm-env && /content/vllm-env/bin/pip install -q vllm
!/content/vllm-env/bin/python -c "import vllm; print('vllm', vllm.__version__)"   # record this version in the run notes

## 4. Data (DVC from DagsHub) and the LLM cache from Drive

In [ ]:
!uv run dvc remote modify --local origin access_key_id "$S3_ACSESS_KEY_ID"
!uv run dvc remote modify --local origin secret_access_key "$S3_SECRET_ACSESS_KEY"
!uv run dvc pull -q
CACHE = "/content/CareFlow/.cache/llm"
if os.path.exists(f"{DRIVE}/cache/llm_cache.sqlite"):
    !uv run python scripts/cache_tool.py snapshot {DRIVE}/cache {CACHE}
!uv run python scripts/cache_tool.py stats {CACHE} 2>/dev/null | head

## 5. Helpers

In [ ]:
def save():
    """Snapshot cache + outputs to Drive (run after every experiment)."""
    !uv run python scripts/cache_tool.py snapshot {CACHE} {DRIVE}/cache
    !rsync -a --exclude logs/vllm outputs/ {DRIVE}/outputs/

def run(config, name, *overrides):
    ov = " ".join(overrides)  # noqa: F841 (used in the shell line)
    !uv run python scripts/run_experiment.py --config {config} --overlay configs/colab.yaml experiment.name={name} {ov}
    save()

def two_phase(config, name, *overrides):
    """systems_eval-style run: generation with gen servers, then judging with the judge server."""
    !GEN_ARGS="{GEN_ARGS}" scripts/colab_serve.sh gen
    run(config, name, "stop_after_generation=true", *overrides)
    !scripts/colab_serve.sh stop
    !JUDGE_ARGS="{JUDGE_ARGS}" scripts/colab_serve.sh judge
    run(config, name, *overrides)
    !scripts/colab_serve.sh stop

## 6. FIRST: validate the judge on this backend
Same pre-registered criteria as on the Mac (Kendall τ with HEART ranking, CI excludes 0; order consistency ≥ 0.8; parse failures < 5%). If they fail, stop: verdicts from this backend are not usable.

In [ ]:
!JUDGE_ARGS="{JUDGE_ARGS}" scripts/colab_serve.sh judge
run("configs/experiments/judge_validation.yaml", "judge_validation_colab")
!scripts/colab_serve.sh stop

## 7. Run everything (≈ 1–1.5 h on A100 incl. model downloads)
Order = priority. Each block is self-contained on this backend (own `single_9b` reference); analysis steps are CPU-only.
1. potolok + placebo (главный вопрос: удача ответа или свойство диалога);
2. replication of the confirmatory design on this backend (other weights → "not an MLX artefact");
3. length-matched variant.

In [ ]:
E = "configs/experiments"
# 1) routing headroom and its noise floor on this backend
two_phase(f"{E}/router_data_test.yaml", "router_data_test_colab")
two_phase(f"{E}/placebo_samples.yaml", "placebo_samples_colab")
run(f"{E}/real_oracle_test.yaml", "real_oracle_test_colab", "source_dir=outputs/router_data_test_colab")
run(f"{E}/placebo_oracle.yaml", "placebo_oracle_colab", "source_dir=outputs/placebo_samples_colab")
# 2) confirmatory design replicated on these weights (the pre-registered run itself stays on the Mac)
two_phase(f"{E}/confirmatory.yaml", "confirmatory_colab", "prereg_files=[]")
run(f"{E}/confirmatory_analysis.yaml", "confirmatory_analysis_colab", "sources=[outputs/confirmatory_colab]")
# 3) length-matched variant
two_phase(f"{E}/confirmatory_len30.yaml", "confirmatory_len30_colab")
run(f"{E}/confirmatory_len30_analysis.yaml", "confirmatory_len30_analysis_colab", "sources=[outputs/confirmatory_len30_colab]")

## 7b. Key numbers (paste this output to the chat)

In [ ]:
import glob
from pathlib import Path

import pandas as pd

for f in ["real_oracle_test_colab", "placebo_oracle_colab"]:
    print(f"== {f}"); print(pd.read_csv(f"outputs/{f}/routes.csv").to_string(index=False))
for f in ["confirmatory_analysis_colab", "confirmatory_len30_analysis_colab"]:
    print(f"== {f}"); d = pd.read_csv(f"outputs/{f}/equivalence.csv")
    print(d[["system", "kind", "n_contexts", "winrate", "verdict@0.1"]].to_string(index=False))
for log in glob.glob("outputs/logs/vllm/*.log"):
    print(log, Path(log).read_text()[-300:])

## 8. Back on the Mac
```bash
python scripts/cache_tool.py merge "<Drive>/careflow/cache" .cache/llm      # adds Colab calls, never overwrites
rsync -a "<Drive>/careflow/outputs/" outputs/
```